In [ ]:
from PIL import Image
import os
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder_path = "/content/drive/MyDrive/Krysztaly_plb/02_training/train_1_plb/pred_unique_jpgs_1601_files_907cd50/crop"

In [ ]:
file_names = os.listdir(folder_path)
image_files = [f for f in file_names if f.endswith('.jpg')]

In [ ]:
df = pd.DataFrame(image_files, columns=["name_PLB"])

In [ ]:
len(df)

3068

In [ ]:
df["img_number"] = df["name_PLB"].str.extract(r'obj(\d+)').astype(int)

In [ ]:
df["raw_png"] = (
    df["name_PLB"]
    .str.replace(r'_obj\d+\.jpg$', '.png', regex=True)
)

In [ ]:
rozdzielczosc = pd.read_csv("/content/drive/MyDrive/Krysztaly_plb/02_training/train_1_plb/pred_unique_jpgs_1601_files_907cd50/raw_png_2.csv")

In [ ]:
rozdzielczosc

,Nazwa pliku,Rozdzielczość (px/nm)
0,00_WTara_1_0_b_40kx__(1).png,0.853
1,01.png,0.636
2,01_WTara_1_0_b_8kx__(1).png,0.165
3,02.png,2.150
4,03.png,0.636
...,...,...
1596,ZS_L_1_0_30kx_6 [1].png,0.633
1597,ZS_L_1_0_40kx_3 [1].png,0.853
1598,ZS_L_1_0_40kx_37 [1].png,0.853
1599,ZS_L_1_0_40kx_45 [1].png,0.853


In [ ]:
resolution_dict = pd.Series(rozdzielczosc['Rozdzielczość (px/nm)'].values,
                            index=rozdzielczosc['Nazwa pliku']).to_dict()

df['resolution_(px/nm)'] = df['raw_png'].map(resolution_dict)

In [ ]:
def get_image_size(file_path):
    try:
        with Image.open(file_path) as img:
            return img.width, img.height
    except Exception as e:
        print(f"error for file  {file_path}: {e}")
        return None, None

In [ ]:
widths = []
heights = []

for file_name in df['name_PLB']:
    full_path = os.path.join(folder_path, file_name)
    w, h = get_image_size(full_path)
    widths.append(w)
    heights.append(h)

df['width'] = widths
df['height'] = heights

In [ ]:
df

,name_PLB,img_number,raw_png,resolution_(px/nm),width,height
0,Col_5d_suc_k_30000x_06_obj1.jpg,1,Col_5d_suc_k_30000x_06.png,0.616,1284,805
1,Col_5d_suc_k_30000x_03_obj1.jpg,1,Col_5d_suc_k_30000x_03.png,0.616,866,858
2,Col_5d_suc_k_30000x_06_obj2.jpg,2,Col_5d_suc_k_30000x_06.png,0.616,400,365
3,Col_5d_suc_k_50000x_11_obj1.jpg,1,Col_5d_suc_k_50000x_11.png,1.015,1030,1162
4,Col_5d_suc_k_50000x_14_obj1.jpg,1,Col_5d_suc_k_50000x_14.png,1.015,1025,1149
...,...,...,...,...,...,...
3063,Col_5d_suc_7O2_30000x_15_obj2.jpg,2,Col_5d_suc_7O2_30000x_15.png,0.616,426,468
3064,Col_5d_suc_7O2_40000x_12_obj2.jpg,2,Col_5d_suc_7O2_40000x_12.png,0.820,1002,901
3065,Col_5d_suc_7O2_40000x_12_obj3.jpg,3,Col_5d_suc_7O2_40000x_12.png,0.820,1155,999
3066,Col_5d_suc_7O2_40000x_12_obj4.jpg,4,Col_5d_suc_7O2_40000x_12.png,0.820,457,522


In [ ]:
def add_yolo_confidence(df, labels_folder):
    """
    Adds a 'score' column to the dataframe by matching YOLO confidence
    from label files based on the object number in the crop name.
    """

    scores = []

    for idx, row in df.iterrows():
        raw_name = row['raw_png']
        crop_name = row['name_PLB']

        # extract object number from crop name
        import re
        match = re.search(r'_obj(\d+)\.jpg$', crop_name)
        if not match:
            scores.append(None)
            continue

        obj_index = int(match.group(1)) - 1  # line index w txt (0-based)

        # path to YOLO label file
        txt_name = os.path.splitext(raw_name)[0] + ".txt"
        txt_path = os.path.join(labels_folder, txt_name)

        if not os.path.exists(txt_path):
            scores.append(None)
            continue

        # read lines from the label file
        with open(txt_path, 'r') as f:
            lines = f.readlines()

        if obj_index >= len(lines):
            scores.append(None)
            continue

        line = lines[obj_index].strip().split()
        if len(line) < 6:
            scores.append(None)
            continue

        try:
            confidence = float(line[5])
        except:
            confidence = None

        scores.append(confidence)

    df['score'] = scores
    return df

In [ ]:
labels_folder = "/content/drive/MyDrive/Krysztaly_plb/02_training/train_1_plb/pred_unique_jpgs_1601_files_907cd50/predict2/labels"

df = add_yolo_confidence(df, labels_folder)


In [ ]:
df.to_csv("/content/drive/MyDrive/Krysztaly_plb/02_training/train_1_plb/pred_unique_jpgs_1601_files_907cd50/data_summary.csv", index=False)

In [ ]:
df

,name_PLB,img_number,raw_png,resolution_(px/nm),width,height,score
0,Col_5d_suc_k_30000x_06_obj1.jpg,1,Col_5d_suc_k_30000x_06.png,0.616,1284,805,0.915194
1,Col_5d_suc_k_30000x_03_obj1.jpg,1,Col_5d_suc_k_30000x_03.png,0.616,866,858,0.920168
2,Col_5d_suc_k_30000x_06_obj2.jpg,2,Col_5d_suc_k_30000x_06.png,0.616,400,365,0.911924
3,Col_5d_suc_k_50000x_11_obj1.jpg,1,Col_5d_suc_k_50000x_11.png,1.015,1030,1162,0.930156
4,Col_5d_suc_k_50000x_14_obj1.jpg,1,Col_5d_suc_k_50000x_14.png,1.015,1025,1149,0.924561
...,...,...,...,...,...,...,...
3063,Col_5d_suc_7O2_30000x_15_obj2.jpg,2,Col_5d_suc_7O2_30000x_15.png,0.616,426,468,0.891517
3064,Col_5d_suc_7O2_40000x_12_obj2.jpg,2,Col_5d_suc_7O2_40000x_12.png,0.820,1002,901,0.911104
3065,Col_5d_suc_7O2_40000x_12_obj3.jpg,3,Col_5d_suc_7O2_40000x_12.png,0.820,1155,999,0.906679
3066,Col_5d_suc_7O2_40000x_12_obj4.jpg,4,Col_5d_suc_7O2_40000x_12.png,0.820,457,522,0.895241
